# 05 — 実データで XGBoost：評価設計もモデルの一部
今回は scikit-learn 同梱の乳がん診断データの二値分類を教材として使う。これは教育目的の分類例であり、臨床判断用のモデルを作るものではない。

**重要:** 学習用 train、学習率・早期終了を決める validation、最後に一度だけ測る test を分離。test を early stopping に使うと、test にも適合してしまう。実データではグループや時系列での分割が必要な場合がある。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
data = load_breast_cancer()
X, y = data.data, data.target
X_trval, X_test, y_trval, y_test = train_test_split(
    X,y,test_size=.20,stratify=y,random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trval,y_trval,test_size=.25,stratify=y_trval,random_state=42)
print("sizes:",len(y_train),len(y_valid),len(y_test))
print("class share:",np.mean(y_train),np.mean(y_valid),np.mean(y_test))

## 早期終了 (early stopping)
validation の logloss が改善しなくなれば訓練を止める。XGBoost sklearn API では \`early_stopping_rounds\` を **コンストラクタ**へ、\`eval_set\` を \`fit\` へ渡す（API の版によって指定場所が変わるため、README の依存条件を参照）。

\`predict_proba\` は保存された best_iteration に対応する範囲の木を自動で使う（この estimator API の挙動）。評価には accuracy だけでなく AUC、logloss を併記する。

In [ ]:
baseline = DummyClassifier(strategy="prior").fit(X_train,y_train)
model = XGBClassifier(
    n_estimators=350, max_depth=3, learning_rate=.04,
    subsample=.9, colsample_bytree=.9,
    reg_lambda=2.0, objective="binary:logistic",
    eval_metric="logloss", tree_method="hist",
    early_stopping_rounds=20, random_state=42, n_jobs=2
)
model.fit(X_train,y_train,eval_set=[(X_valid,y_valid)],verbose=False)
for name,m in [("Dummy",baseline),("XGBoost",model)]:
    proba = m.predict_proba(X_test)[:,1]
    print(f"{name}: accuracy={accuracy_score(y_test, proba>=.5):.3f}, AUC={roc_auc_score(y_test,proba):.3f}, logloss={log_loss(y_test,proba):.3f}")
print("Best iteration (0-index):",model.best_iteration)

In [ ]:
history=model.evals_result()["validation_0"]["logloss"]
plt.plot(history)
plt.axvline(model.best_iteration,color="black",linestyle="--",label="best iteration")
plt.xlabel("boosting round (0-index)")
plt.ylabel("validation logloss")
plt.title("Validation learning curve")
plt.legend();plt.show()

## 演習（設計を問う）
1. \`max_depth\` を 1, 3, 6 で変え、**validation** のみを使って比較。test の値で選び直さない。
2. \`reg_lambda\` を変え、木の複雑性・予測結果の関係を説明。
3. \`learning_rate\` を下げたとき、必要な \`n_estimators\` の上限はどうなる？
4. 最後に一度だけ test を計算。サンプルサイズと指標の不確実性について一言書く。

**注意:** 木ベースで標準化は一般に不要。ただし「すべての前処理が不要」ではない。欠損・カテゴリ・リーク・分布変動には別途配慮が要る。